# FlowOS V2: Remote SFT -> GRPO on Hugging Face Jobs

This notebook launches remote HF Jobs instead of training on Kaggle GPU.

Pipeline:
1. Collect ranked traces remotely
2. Train SFT remotely on `l4x1`
3. Upload the SFT checkpoint to HF Hub
4. Launch GRPO from that SFT checkpoint
5. Monitor or cancel jobs from this notebook

In [ ]:
!pip install -q -U huggingface_hub

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("HF auth ready")

In [ ]:
from huggingface_hub import HfApi

REPO_URL = "https://github.com/pranjalparashar/FlowOS_v2.git"
ENV_URL = "https://praanjal-flowos-v2.hf.space"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_MODEL_REPO = "praanjal/flowos-sft-ranked"
GRPO_RUNS_REPO = "praanjal/flowos-grpo-runs"
HF_FLAVOR = "l4x1"

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=SFT_MODEL_REPO, repo_type="model", exist_ok=True)
api.create_repo(repo_id=GRPO_RUNS_REPO, repo_type="dataset", exist_ok=True)
print("Output repos are ready")

## Launch ranked SFT

In [ ]:
from huggingface_hub import run_job

sft_job = run_job(
    image="python:3.11",
    flavor=HF_FLAVOR,
    timeout="8h",
    command=[
        "bash",
        "-lc",
        f"""
        set -e
        export HF_TOKEN=\"{HF_TOKEN}\"
        git clone {REPO_URL} repo
        cd repo
        pip install -U pip
        pip install -e '.[train]'

        python collect_traces.py \
          --env-url {ENV_URL} \
          --task-scope simulate_csv_report_workflow \
          --dataset-size 56 \
          --max-turns 12 \
          --output-dir outputs/sft_traces_ranked

        CUDA_VISIBLE_DEVICES=0 python train_sft.py \
          --dataset-path outputs/sft_traces_ranked/traces.jsonl \
          --model-id {BASE_MODEL} \
          --output-dir outputs/flowos-sft-ranked \
          --num-epochs 2 \
          --per-device-train-batch-size 1 \
          --gradient-accumulation-steps 4 \
          --max-seq-length 1024 \
          --load-in-4bit \
          --gradient-checkpointing \
          --min-trace-rank 0.55 \
          --top-trace-fraction 0.6

        python - <<'PY'
import os
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
api.upload_folder(
    repo_id='{SFT_MODEL_REPO}',
    repo_type='model',
    folder_path='outputs/flowos-sft-ranked',
)
print('Uploaded {SFT_MODEL_REPO}')
PY
        """
    ],
)

print("SFT Job ID:", sft_job.id)
print("SFT Job URL:", sft_job.url)

## Monitor a job

In [ ]:
from huggingface_hub import inspect_job

job_to_check = sft_job.id
info = inspect_job(job_to_check)
print(info.status)
print(info.url)

In [ ]:
from huggingface_hub import fetch_job_logs

job_to_check = sft_job.id
for log in fetch_job_logs(job_to_check):
    print(log, end="")

## Launch GRPO from the uploaded SFT checkpoint

Run this only after the SFT upload has completed successfully.

In [ ]:
from huggingface_hub import run_job

grpo_job = run_job(
    image="python:3.11",
    flavor=HF_FLAVOR,
    timeout="8h",
    command=[
        "bash",
        "-lc",
        f"""
        set -e
        export HF_TOKEN=\"{HF_TOKEN}\"
        git clone {REPO_URL} repo
        cd repo
        pip install -U pip
        pip install -e '.[train]'

        python train.py \
          --model-id {BASE_MODEL} \
          --init-checkpoint {SFT_MODEL_REPO} \
          --env-url {ENV_URL} \
          --task-scope simulate_csv_report_workflow \
          --dataset-size 28 \
          --num-generations 2 \
          --num-epochs 1 \
          --max-turns 12 \
          --max-new-tokens 160 \
          --per-device-train-batch-size 1 \
          --gradient-accumulation-steps 4 \
          --save-steps 10 \
          --logging-steps 1 \
          --report-to none \
          --fallback-mode never \
          --upload-repo-id {GRPO_RUNS_REPO} \
          --upload-repo-type dataset
        """
    ],
)

print("GRPO Job ID:", grpo_job.id)
print("GRPO Job URL:", grpo_job.url)

## Cancel a job

In [ ]:
from huggingface_hub import HfApi

JOB_ID_TO_CANCEL = ""
if JOB_ID_TO_CANCEL:
    HfApi(token=HF_TOKEN).cancel_job(job_id=JOB_ID_TO_CANCEL)
    print("Canceled", JOB_ID_TO_CANCEL)
else:
    print("Set JOB_ID_TO_CANCEL first")